# Integrate Knowledge

Integrates validated structured metadata and validated unstructured content from Silver into a unified Gold knowledge model. The resulting knowledge items provide a common representation for downstream chunking, embeddings, and retrieval.

## Imports

In [2]:
from pyspark.sql import functions as F

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 4, Finished, Available, Finished, False)

## Load Silver Sources

Loads validated structured metadata and standardized document content from Silver as trusted inputs for Gold knowledge integration.

In [ ]:
def load_silver_sources():
    """
    Loads the Silver datasets required for knowledge integration
    and exposes their direct Silver execution lineage.
    """

    silver_sources = {}

    silver_sources["content_documents"] = (
        spark.table("silver.content_documents")
        .select(
            "content_id",
            "source_container",
            "source_path",
            "file_name",
            "content",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    silver_sources["metadata_objects"] = (
        spark.table("silver.metadata_objects")
        .select(
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "object_type",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    silver_sources["metadata_columns"] = (
        spark.table("silver.metadata_columns")
        .select(
            "column_id",
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "column_name",
            "data_type",
            "is_nullable",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    silver_sources["metadata_constraints"] = (
        spark.table("silver.metadata_constraints")
        .select(
            "constraint_id",
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "constraint_name",
            "constraint_type",
            "referenced_object_id",
            "referenced_database",
            "referenced_schema",
            "referenced_object",
            "delete_action",
            "update_action",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    silver_sources["metadata_constraint_columns"] = (
        spark.table("silver.metadata_constraint_columns")
        .select(
            "constraint_id",
            "column_id",
            "column_name",
            "referenced_column_name",
            "ordinal_position",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    silver_sources["metadata_relationships"] = (
        spark.table("silver.metadata_relationships")
        .select(
            "relationship_id",
            "source_database",
            "source_schema",
            "source_object",
            "source_column",
            "target_database",
            "target_schema",
            "target_object",
            "target_column",
            "relationship_type",
            "relationship_name",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    silver_sources["metadata_modules"] = (
        spark.table("silver.metadata_modules")
        .select(
            "module_id",
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "object_type",
            "module_definition",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    return silver_sources

## Integrate Document Knowledge

Transforms validated Silver document content into unified knowledge candidates with stable identity, source context, direct Silver lineage, and original extracted text.

In [5]:
def integrate_document_knowledge(
    silver_content_documents
):
    """
    Transforms validated Silver content documents into unified
    knowledge candidates.
    """

    return (
        silver_content_documents
        .select(
            F.col("content_id").alias("knowledge_id"),
            F.lit("DOCUMENT").alias("knowledge_type"),
            F.lit("UNSTRUCTURED_CONTENT").alias("source_type"),
            F.col("source_container").alias("source_scope"),
            F.concat_ws(
                "/",
                F.col("source_container"),
                F.col("source_path")
            ).alias("source_name"),
            F.col("file_name").alias("title"),
            F.col("content"),
            F.col("source_path"),
            F.lit(None)
                .cast("string")
                .alias("source_object_id"),
            F.col("source_execution_step_id")
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 7, Finished, Available, Finished, False)

## Integrate Metadata Object Knowledge

Transforms validated Silver metadata objects into unified knowledge candidates with source context, object identity, descriptive technical content, and direct Silver lineage.

In [6]:
def integrate_object_knowledge(
    silver_metadata_objects
):
    """
    Transforms validated Silver metadata objects into unified
    knowledge candidates.
    """

    return (
        silver_metadata_objects
        .select(
            F.col("object_id").alias("knowledge_id"),
            F.lit("DATABASE_OBJECT").alias("knowledge_type"),
            F.lit("STRUCTURED_METADATA").alias("source_type"),
            F.col("source_database").alias("source_scope"),
            F.concat_ws(
                ".",
                F.col("source_database"),
                F.col("source_schema"),
                F.col("object_name")
            ).alias("source_name"),
            F.concat_ws(
                ".",
                F.col("source_schema"),
                F.col("object_name")
            ).alias("title"),
            F.concat_ws(
                " | ",
                F.concat(
                    F.lit("Database: "),
                    F.col("source_database")
                ),
                F.concat(
                    F.lit("Schema: "),
                    F.col("source_schema")
                ),
                F.concat(
                    F.lit("Object: "),
                    F.col("object_name")
                ),
                F.concat(
                    F.lit("Type: "),
                    F.col("object_type")
                )
            ).alias("content"),
            F.lit(None)
                .cast("string")
                .alias("source_path"),
            F.col("object_id").alias("source_object_id"),
            F.col("source_execution_step_id")
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 8, Finished, Available, Finished, False)

## Integrate Metadata Column Knowledge

Transforms validated Silver column metadata into unified knowledge candidates with source context, parent object identity, technical attributes, and direct Silver lineage.

In [7]:
def integrate_column_knowledge(
    silver_metadata_columns
):
    """
    Transforms validated Silver metadata columns into unified
    knowledge candidates.
    """

    return (
        silver_metadata_columns
        .select(
            F.col("column_id").alias("knowledge_id"),
            F.lit("COLUMN").alias("knowledge_type"),
            F.lit("STRUCTURED_METADATA").alias("source_type"),
            F.col("source_database").alias("source_scope"),
            F.concat_ws(
                ".",
                F.col("source_database"),
                F.col("source_schema"),
                F.col("object_name"),
                F.col("column_name")
            ).alias("source_name"),
            F.concat_ws(
                ".",
                F.col("object_name"),
                F.col("column_name")
            ).alias("title"),
            F.concat_ws(
                " | ",
                F.concat(
                    F.lit("Database: "),
                    F.col("source_database")
                ),
                F.concat(
                    F.lit("Schema: "),
                    F.col("source_schema")
                ),
                F.concat(
                    F.lit("Object: "),
                    F.col("object_name")
                ),
                F.concat(
                    F.lit("Column: "),
                    F.col("column_name")
                ),
                F.concat(
                    F.lit("Data Type: "),
                    F.col("data_type")
                ),
                F.concat(
                    F.lit("Nullable: "),
                    F.col("is_nullable").cast("string")
                )
            ).alias("content"),
            F.lit(None)
                .cast("string")
                .alias("source_path"),
            F.col("object_id").alias("source_object_id"),
            F.col("source_execution_step_id")
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 9, Finished, Available, Finished, False)

## Integrate Metadata Constraint Knowledge

Transforms validated Silver constraint metadata into unified knowledge candidates, enriching each constraint with its participating columns, referenced object and columns when applicable, referential actions, source context, parent object identity, and direct Silver lineage.

In [8]:
def integrate_constraint_knowledge(
    silver_metadata_constraints,
    silver_metadata_constraint_columns
):
    """
    Transforms validated Silver constraint metadata into unified
    knowledge candidates enriched with participating columns.
    """

    constraint_columns = (
        silver_metadata_constraint_columns
        .groupBy("constraint_id")
        .agg(
            F.concat_ws(
                ", ",
                F.transform(
                    F.sort_array(
                        F.collect_list(
                            F.struct(
                                F.col("ordinal_position"),
                                F.col("column_name")
                            )
                        )
                    ),
                    lambda x: x["column_name"]
                )
            ).alias("columns"),
            F.concat_ws(
                ", ",
                F.transform(
                    F.sort_array(
                        F.collect_list(
                            F.struct(
                                F.col("ordinal_position"),
                                F.col("referenced_column_name")
                            )
                        )
                    ),
                    lambda x: x["referenced_column_name"]
                )
            ).alias("referenced_columns")
        )
    )

    enriched_constraints = (
        silver_metadata_constraints.alias("c")
        .join(
            constraint_columns.alias("cc"),
            F.col("c.constraint_id") == F.col("cc.constraint_id"),
            "left"
        )
    )

    return (
        enriched_constraints
        .select(
            F.col("c.constraint_id").alias("knowledge_id"),
            F.lit("CONSTRAINT").alias("knowledge_type"),
            F.lit("STRUCTURED_METADATA").alias("source_type"),
            F.col("c.source_database").alias("source_scope"),
            F.concat_ws(
                ".",
                F.col("c.source_database"),
                F.col("c.source_schema"),
                F.col("c.object_name"),
                F.col("c.constraint_name")
            ).alias("source_name"),
            F.concat_ws(
                ".",
                F.col("c.object_name"),
                F.col("c.constraint_name")
            ).alias("title"),
            F.concat_ws(
                " | ",
                F.concat(
                    F.lit("Database: "),
                    F.col("c.source_database")
                ),
                F.concat(
                    F.lit("Schema: "),
                    F.col("c.source_schema")
                ),
                F.concat(
                    F.lit("Object: "),
                    F.col("c.object_name")
                ),
                F.concat(
                    F.lit("Constraint: "),
                    F.col("c.constraint_name")
                ),
                F.concat(
                    F.lit("Type: "),
                    F.col("c.constraint_type")
                ),
                F.when(
                    F.col("cc.columns").isNotNull(),
                    F.concat(
                        F.lit("Columns: "),
                        F.col("cc.columns")
                    )
                ),
                F.when(
                    F.col("c.referenced_object_id").isNotNull(),
                    F.concat(
                        F.lit("Referenced Object: "),
                        F.concat_ws(
                            ".",
                            F.col("c.referenced_database"),
                            F.col("c.referenced_schema"),
                            F.col("c.referenced_object")
                        )
                    )
                ),
                F.when(
                    F.col("cc.referenced_columns").isNotNull(),
                    F.concat(
                        F.lit("Referenced Columns: "),
                        F.col("cc.referenced_columns")
                    )
                ),
                F.when(
                    F.col("c.delete_action").isNotNull(),
                    F.concat(
                        F.lit("Delete Action: "),
                        F.col("c.delete_action")
                    )
                ),
                F.when(
                    F.col("c.update_action").isNotNull(),
                    F.concat(
                        F.lit("Update Action: "),
                        F.col("c.update_action")
                    )
                )
            ).alias("content"),
            F.lit(None)
                .cast("string")
                .alias("source_path"),
            F.col("c.object_id").alias("source_object_id"),
            F.col("c.source_execution_step_id")
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 10, Finished, Available, Finished, False)

## Integrate Metadata Relationship Knowledge

Transforms validated Silver relationship metadata into unified knowledge candidates describing the source and target objects and columns, relationship type and name, source context, and direct Silver lineage.

In [11]:
def integrate_relationship_knowledge(
    silver_metadata_relationships
):
    """
    Transforms validated Silver relationship metadata into unified
    knowledge candidates.
    """

    return (
        silver_metadata_relationships
        .select(
            F.col("relationship_id").alias("knowledge_id"),
            F.lit("RELATIONSHIP").alias("knowledge_type"),
            F.lit("STRUCTURED_METADATA").alias("source_type"),
            F.col("source_database").alias("source_scope"),
            F.concat(
                F.concat_ws(
                    ".",
                    F.col("source_database"),
                    F.col("source_schema"),
                    F.col("source_object"),
                    F.col("source_column")
                ),
                F.lit(" -> "),
                F.concat_ws(
                    ".",
                    F.col("target_database"),
                    F.col("target_schema"),
                    F.col("target_object"),
                    F.col("target_column")
                )
            ).alias("source_name"),
            F.concat(
                F.concat_ws(
                    ".",
                    F.col("source_object"),
                    F.col("source_column")
                ),
                F.lit(" -> "),
                F.concat_ws(
                    ".",
                    F.col("target_object"),
                    F.col("target_column")
                )
            ).alias("title"),
            F.concat_ws(
                " | ",
                F.concat(
                    F.lit("Source: "),
                    F.concat_ws(
                        ".",
                        F.col("source_database"),
                        F.col("source_schema"),
                        F.col("source_object"),
                        F.col("source_column")
                    )
                ),
                F.concat(
                    F.lit("Target: "),
                    F.concat_ws(
                        ".",
                        F.col("target_database"),
                        F.col("target_schema"),
                        F.col("target_object"),
                        F.col("target_column")
                    )
                ),
                F.concat(
                    F.lit("Relationship Type: "),
                    F.col("relationship_type")
                ),
                F.when(
                    F.col("relationship_name").isNotNull(),
                    F.concat(
                        F.lit("Relationship Name: "),
                        F.col("relationship_name")
                    )
                )
            ).alias("content"),
            F.lit(None)
                .cast("string")
                .alias("source_path"),
            F.lit(None)
                .cast("string")
                .alias("source_object_id"),
            F.col("source_execution_step_id")
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 13, Finished, Available, Finished, False)

## Integrate Metadata Module Knowledge

Transforms validated Silver module metadata into unified knowledge candidates with source context, parent object identity, module type, SQL definition, and direct Silver lineage.

In [12]:
def integrate_module_knowledge(
    silver_metadata_modules
):
    """
    Transforms validated Silver module metadata into unified
    knowledge candidates.
    """

    return (
        silver_metadata_modules
        .select(
            F.col("module_id").alias("knowledge_id"),
            F.lit("MODULE").alias("knowledge_type"),
            F.lit("STRUCTURED_METADATA").alias("source_type"),
            F.col("source_database").alias("source_scope"),
            F.concat_ws(
                ".",
                F.col("source_database"),
                F.col("source_schema"),
                F.col("object_name")
            ).alias("source_name"),
            F.concat_ws(
                ".",
                F.col("source_schema"),
                F.col("object_name")
            ).alias("title"),
            F.concat_ws(
                "\n\n",
                F.concat(
                    F.lit("Database: "),
                    F.col("source_database")
                ),
                F.concat(
                    F.lit("Schema: "),
                    F.col("source_schema")
                ),
                F.concat(
                    F.lit("Object: "),
                    F.col("object_name")
                ),
                F.concat(
                    F.lit("Type: "),
                    F.col("object_type")
                ),
                F.concat(
                    F.lit("Definition:\n"),
                    F.col("module_definition")
                )
            ).alias("content"),
            F.lit(None)
                .cast("string")
                .alias("source_path"),
            F.col("object_id").alias("source_object_id"),
            F.col("source_execution_step_id")
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 14, Finished, Available, Finished, False)

## Combine Integrated Knowledge

Combines document and structured metadata knowledge candidates into a single unified dataset using the common `knowledge_items` schema.

In [13]:
def combine_knowledge_items(
    document_knowledge,
    object_knowledge,
    column_knowledge,
    constraint_knowledge,
    relationship_knowledge,
    module_knowledge
):
    """
    Combines all knowledge candidate datasets into the common
    knowledge_items schema.
    """

    knowledge_columns = [
        "knowledge_id",
        "knowledge_type",
        "source_type",
        "source_scope",
        "source_name",
        "title",
        "content",
        "source_path",
        "source_object_id",
        "source_execution_step_id"
    ]

    return (
        document_knowledge.select(knowledge_columns)
        .unionByName(
            object_knowledge.select(knowledge_columns)
        )
        .unionByName(
            column_knowledge.select(knowledge_columns)
        )
        .unionByName(
            constraint_knowledge.select(knowledge_columns)
        )
        .unionByName(
            relationship_knowledge.select(knowledge_columns)
        )
        .unionByName(
            module_knowledge.select(knowledge_columns)
        )
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 15, Finished, Available, Finished, False)

## Write Integrated Knowledge to Work

Replaces the current integrated knowledge candidate set in the Work layer before validation and promotion to Gold.

In [15]:
def write_work_knowledge(
    work_knowledge_items
):
    """
    Replaces the current Work knowledge candidate snapshot.
    """

    (
        work_knowledge_items.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("work.knowledge_items")
    )

StatementMeta(, 5157b501-fbaa-4b02-961f-e03b78650e88, 17, Finished, Available, Finished, False)

## Execute Knowledge Integration Process

Orchestrates Silver source loading, knowledge transformation, unification, and Work publication for the current Gold integration cycle.

In [ ]:
silver_sources = load_silver_sources()

work_document_knowledge = integrate_document_knowledge(
    silver_content_documents=silver_sources["content_documents"]
)

work_object_knowledge = integrate_object_knowledge(
    silver_metadata_objects=silver_sources["metadata_objects"]
)

work_column_knowledge = integrate_column_knowledge(
    silver_metadata_columns=silver_sources["metadata_columns"]
)

work_constraint_knowledge = integrate_constraint_knowledge(
    silver_metadata_constraints=silver_sources["metadata_constraints"],
    silver_metadata_constraint_columns=silver_sources["metadata_constraint_columns"]
)

work_relationship_knowledge = integrate_relationship_knowledge(
    silver_metadata_relationships=silver_sources["metadata_relationships"]
)

work_module_knowledge = integrate_module_knowledge(
    silver_metadata_modules=silver_sources["metadata_modules"]
)

work_knowledge_items = combine_knowledge_items(
    document_knowledge=work_document_knowledge,
    object_knowledge=work_object_knowledge,
    column_knowledge=work_column_knowledge,
    constraint_knowledge=work_constraint_knowledge,
    relationship_knowledge=work_relationship_knowledge,
    module_knowledge=work_module_knowledge
)

write_work_knowledge(
    work_knowledge_items=work_knowledge_items
)